In [22]:
import numpy as np
from unified_planning.shortcuts import *
from unified_planning.engines import PlanGenerationResultStatus
import torch
import tqdm
import random

from unified_planning.shortcuts import Compiler, SequentialSimulator
from unified_planning.engines import CompilationKind
from unified_planning.plans import ActionInstance

In [25]:
blocksworld_config = {
    "blocks": ["A", "B", "C"],
    "initial_state": {
        ("ontable", "A"): True,
        ("ontable", "B"): True,
        ("ontable", "C"): True,
        ("clear", "A"): True,
        ("clear", "B"): True,
        ("clear", "C"): True,
        ("handempty",): True,
    },
    "goal_state": {
        ("on", "A", "B"): True,
        ("on", "B", "C"): True,
    },
}

problem = create_blocksworld_problem(blocksworld_config)

data = enumerate_trajectories_up_to_horizon(
    problem=problem,
    horizon=6,
    k=1000,
    seed=42,
)

  *** Credits ***
  * In operation mode `Compiler` at line 180 of `/tmp/ipykernel_136253/3823043928.py`, you are using the following planning engine:
  * Engine name: Fast Downward
  * Developers:  Uni Basel team and contributors (cf. https://github.com/aibasel/downward/blob/main/README.md)
  * Description: Fast Downward is a domain-independent classical planning system.



In [26]:
print("Plans:")
print(data["plans_by_length"])

print("Non-plan trajectories:")
print(data["trajectories_by_length"])

Plans:
{0: 0, 1: 0, 2: 0, 3: 0, 4: 1, 5: 0, 6: 7}
Non-plan trajectories:
{0: 0, 1: 3, 2: 9, 3: 21, 4: 56, 5: 123, 6: 320}


In [27]:
for plan in data["plans"]:
    print(plan)

[['pick_up', 'A', None], ['put_down', 'A', None], ['pick_up', 'B', None], ['stack', 'B', 'C'], ['pick_up', 'A', None], ['stack', 'A', 'B'], ['END', None, None]]
[['pick_up', 'B', None], ['put_down', 'B', None], ['pick_up', 'B', None], ['stack', 'B', 'C'], ['pick_up', 'A', None], ['stack', 'A', 'B'], ['END', None, None]]
[['pick_up', 'B', None], ['stack', 'B', 'A'], ['unstack', 'B', 'A'], ['stack', 'B', 'C'], ['pick_up', 'A', None], ['stack', 'A', 'B'], ['END', None, None]]
[['pick_up', 'B', None], ['stack', 'B', 'C'], ['pick_up', 'A', None], ['put_down', 'A', None], ['pick_up', 'A', None], ['stack', 'A', 'B'], ['END', None, None]]
[['pick_up', 'B', None], ['stack', 'B', 'C'], ['pick_up', 'A', None], ['stack', 'A', 'B'], ['END', None, None], ['PAD', None, None], ['PAD', None, None]]
[['pick_up', 'B', None], ['stack', 'B', 'C'], ['pick_up', 'A', None], ['stack', 'A', 'B'], ['unstack', 'A', 'B'], ['stack', 'A', 'B'], ['END', None, None]]
[['pick_up', 'B', None], ['stack', 'B', 'C'], ['uns

In [67]:
problem = create_blocksworld_problem(blocksworld_config)

H = 6

codec = PlanCodec.from_problem(
    problem,
    horizon=H,
)

codec.print_vocabulary()

  0: <PAD>
  1: <MASK>
  2: <NONE>
  3: <END>
  4: <INIT>
  5: <GOAL>
  6: pick_up
  7: put_down
  8: stack
  9: unstack
 10: clear
 11: handempty
 12: holding
 13: on
 14: ontable
 15: A
 16: B
 17: C


In [68]:
plan = [
    ["pick_up", "B", None],
    ["stack", "B", "C"],
    ["pick_up", "A", None],
    ["stack", "A", "B"],
]

In [69]:
encoded = codec.encode_plan(plan)

print(encoded.tensor)
print(encoded.tensor.shape)
print(encoded.length)

tensor([[ 6, 16,  2],
        [ 8, 16, 17],
        [ 6, 15,  2],
        [ 8, 15, 16],
        [ 3,  2,  2],
        [ 0,  0,  0],
        [ 0,  0,  0]])
torch.Size([7, 3])
4


In [70]:
context = (
    codec.encode_problem_context(
        problem
    )
)

print(context.tensor.shape)
print(context.attention_mask.shape)

torch.Size([40, 3])
torch.Size([40])


In [50]:
decoded = codec.decode_plan(encoded.tensor)

print(decoded)

[['pick_up', 'B', None], ['stack', 'B', 'C'], ['pick_up', 'A', None], ['stack', 'A', 'B']]


In [51]:
original = [
    ["pick_up", "B", None],
    ["stack", "B", "C"],
    ["pick_up", "A", None],
    ["stack", "A", "B"],
]

encoded = codec.encode_plan(original)

decoded = codec.decode_plan(encoded.tensor)

reencoded = codec.encode_plan(decoded)

print(decoded)
print(torch.equal(
    encoded.tensor,
    reencoded.tensor,
))

[['pick_up', 'B', None], ['stack', 'B', 'C'], ['pick_up', 'A', None], ['stack', 'A', 'B']]
True


In [ ]:
raw_data = enumerate_trajectories_up_to_horizon(
    problem=problem,
    horizon=6,
    k=1000,
    seed=42,
)

codec = PlanCodec.from_problem(
    problem,
    horizon=6,
)

data = encode_enumeration_result(
    raw_data,
    codec,
)

print(data["plans"].shape)
print(data["trajectories"].shape)

  *** Credits ***
  * In operation mode `Compiler` at line 180 of `/tmp/ipykernel_136253/3823043928.py`, you are using the following planning engine:
  * Engine name: Fast Downward
  * Developers:  Uni Basel team and contributors (cf. https://github.com/aibasel/downward/blob/main/README.md)
  * Description: Fast Downward is a domain-independent classical planning system.

torch.Size([8, 7, 3])
torch.Size([532, 7, 3])


In [56]:
x0 = codec.masked_plan()

In [57]:
x0

tensor([[1, 1, 1],
        [1, 1, 1],
        [1, 1, 1],
        [1, 1, 1],
        [1, 1, 1],
        [1, 1, 1],
        [1, 1, 1]])

PlanValidationResult(valid=True, solves_problem=True, plan=[['pick_up', 'A'], ['put_down', 'A'], ['pick_up', 'B'], ['stack', 'B', 'C'], ['pick_up', 'A'], ['stack', 'A', 'B']], executed_actions=6, failure_step=None, reason='Plan is valid and satisfies the goal.', final_state={clear(A): true, handempty: true, on(A, B): true, on(B, C): true, ontable(C): true})
